<a href="https://colab.research.google.com/github/Syed-Irtaza/CrewLogix_GenAI_Assignment/blob/main/Small_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [7]:
from openai import OpenAI
import numpy as np

client = OpenAI()

In [8]:
documents = [
    "RAG stands for Retrieval Augmented Generation. It retrieves relevant information before generating an answer.",
    "Fine-tuning trains a model on custom examples to change its behavior or style.",
    "RAG is useful when answers depend on documents, PDFs, company data, or frequently changing knowledge.",
    "Embeddings convert text into numerical vectors so similar meanings can be compared.",
    "A vector database stores embeddings and helps retrieve the most relevant chunks."
]

In [9]:
def get_embedding(text):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding

doc_embeddings = [get_embedding(doc) for doc in documents]

In [10]:
def cosine_similarity(a, b):
    a = np.array(a)
    b = np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [11]:
def retrieve(query, top_k=2):
    query_embedding = get_embedding(query)

    scores = [
        cosine_similarity(query_embedding, doc_embedding)
        for doc_embedding in doc_embeddings
    ]

    top_indexes = np.argsort(scores)[-top_k:][::-1]

    return [documents[i] for i in top_indexes]

In [12]:
def rag_answer(query):
    retrieved_docs = retrieve(query)

    context = "\n".join(retrieved_docs)

    prompt = f"""
You are a helpful AI assistant.
Answer the question using only the context below.

Context:
{context}

Question:
{query}
"""

    response = client.responses.create(
        model="gpt-4o-mini",
        input=prompt
    )

    return response.output_text

In [13]:
question = "When should I use RAG instead of fine-tuning?"

answer = rag_answer(question)

print(answer)

You should use RAG instead of fine-tuning when answers depend on documents, PDFs, company data, or frequently changing knowledge.
